# Sophia AGI — LoRA benchmark eval (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomyimkc/sophia-agi/blob/main/notebooks/Sophia-LoRA-Eval-Colab.ipynb)

Run **23 held-out benchmark cases** + optional **runtime gate** on your trained adapter.

1. **Runtime** → **T4 GPU** → **Restart session**
2. Run cells in order
3. Upload `sophia-lora-v1.zip` when prompted (or skip if you trained in the train notebook this session)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime → Change runtime type → T4 GPU → Restart")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import shutil
import subprocess

REPO = "sophia-agi"
REMOTE = "https://github.com/tomyimkc/sophia-agi.git"

if os.path.isdir(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", REMOTE], check=True)
%cd sophia-agi
!git log -1 --oneline

In [ ]:
!pip install -q -U \
  "transformers>=4.44,<5.0" \
  "peft>=0.10,<0.18" \
  "datasets>=2.18" \
  "accelerate>=0.28" \
  "bitsandbytes>=0.43"

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

CKPT = Path("training/lora/checkpoints/sophia-v1")
REQUIRED = ["adapter_config.json", "adapter_model.safetensors"]

missing = [f for f in REQUIRED if not (CKPT / f).exists()]
if missing:
    print("Upload sophia-lora-v1.zip from your PC...")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    if CKPT.exists():
        import shutil
        shutil.rmtree(CKPT)
    CKPT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name, "r") as zf:
        zf.extractall(CKPT)
    print("Extracted to", CKPT)

missing = [f for f in REQUIRED if not (CKPT / f).exists()]
if missing:
    raise RuntimeError(f"Adapter still missing {missing}")
print("Adapter OK:", sorted(p.name for p in CKPT.iterdir()))

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "tools/eval_local_model.py",
    "--adapter", "training/lora/checkpoints/sophia-v1",
    "--4bit",
    "--with-gate",
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, check=False)
if result.returncode not in (0, 1):
    raise RuntimeError(f"eval failed (exit {result.returncode})")
print("Note: exit 1 means some benchmark cases failed — check reports below.")

In [ ]:
import json
from pathlib import Path
import shutil
from google.colab import files

runs = sorted(Path("benchmark/model_runs").glob("local-sophia-v1-*.report.json"))
if not runs:
    raise RuntimeError("No report files found — eval cell did not complete")

for path in runs:
    report = json.loads(path.read_text(encoding="utf-8"))
    gate = report.get("gateFailures", 0)
    print(f"{report['domain']}: {report['passed']}/{report['total']} ({report['score_pct']}%) gate_failures={gate}")

bundle = Path("sophia-eval-reports.zip")
if bundle.exists():
    bundle.unlink()
shutil.make_archive("sophia-eval-reports", "zip", "benchmark/model_runs")
files.download("sophia-eval-reports.zip")

## Optional: upload adapter to Hugging Face

In a local shell (with `HF_TOKEN` set):

```bash
python tools/upload_huggingface_adapter.py --approve
```

Model repo: `tomyimkc/sophia-agi-lora-v1`